# enVector API Flow Example with KMS

This example shows the end-to-end workflow of **enVector** Python SDK (`pyenvector`) with the **enVector KMS** managing the keys.

In this notebook, we'll show the following works step-by-step:
1. how vector encryption works using fully homomorphic encryption (FHE)
1. how encrypted similarity search works on the FHE-encrypted index
3. how decryption is performed by the KMS on the client's behalf

by running the APIs explicitly.
This example ensures that the server only ever sees ciphertexts and the evaluation key, while the secret key and the metadata key never leave the KMS.

## Import SDK

First, you should install and import the `pyenvector` package to use enVector Python APIs.
Before installing, make sure you have Python 3.11 and a virtual environment on your system. 

For more details, see [SDK installation](https://docs.envector.io/get-started/installation/client-sdk) section in [enVector Documents](https://docs.envector.io/).

In [ ]:
import pyenvector as ev

### Connect to the enVector endpoint server and the KMS

First, we open a gRPC connection to the enVector endpoint server with `init_connect()`. Pass `address` (e.g. `host:port`) and an `access_token` if the server requires bearer authentication.

`init_connect()` only sets up the network channel. We then call `init_kms_connect()` to open a second connection to the **enVector KMS** service, which manages the keys; pass `kms_address` and (for a TLS deployment) the CA bundle via `ca_cert`. The KMS connection reuses the same `access_token`.

The high-level `ev.init()` can do both connections, build the `IndexConfig`, and (when `auto_key_setup=True`) generate and register the KMS-managed keys in one shot.

Here we keep these steps separate so we can run each API explicitly.

For more details, see [Connection Configuration](https://docs.envector.io/user-guide/initialize/connection-configuration) section in [enVector Documents](https://docs.envector.io/).

In [ ]:
# Connect to the enVector end-point server with address and access token (if required)
import os
address = os.getenv("ENVECTOR_ADDRESS", "localhost:50050")

# Fetch a Keycloak token in-notebook; `!cmd` captures stdout and the token is the last line.
_out = !../../deployment/scripts/auth/get_keycloak_token.sh app-a password
access_token = _out[-1].strip()

ev.init_connect(
    address=address,
    access_token=access_token,
    secure=False,
)

In [ ]:
kms_address = os.getenv("KMS_ADDRESS", "localhost:50090")
kms_secure = os.getenv("KMS_SECURE", "false").lower() == "true"
kms_ca_cert = os.getenv("KMS_CA_CERT", "root_ca.crt") if kms_secure else None

ev.init_kms_connect(
    kms_address=kms_address,
    secure=kms_secure,
    ca_cert=kms_ca_cert,
    access_token=access_token,
)

In [ ]:
# Or you can call ev.init() for connection and configuration without key registration
# ev.init(address="localhost:50050", key_path=key_path, auto_key_setup=False)

## Index Control

### Index Configuration

We first configure the `IndexConfig`, which tells the SDK which KMS-managed key to use (`key_id`) and which encryption modes to apply.

The cell below sets only the essentials and lets the SDK fill in defaults:
- `metadata_encryption=True` — metadata will be encrypted through the KMS before insert.
- `auto_key_setup=False` — skip automatic key generation/registration; we'll call those steps explicitly below.

For more details, see [Index Config](https://docs.envector.io/user-guide/initialize/index-configuration/4.-indexconfig) in [enVector Documents](https://docs.envector.io/).

In [ ]:
key_id = "e2e_key"
preset = "ip2"      # must match the KMS server's configured preset
eval_mode = "mm32"  # must match the KMS server's configured eval mode

In [ ]:
# Configure index and key config
ev.init_index_config(
    key_id=key_id, 
    preset=preset, 
    eval_mode=eval_mode,
    metadata_encryption=True,
    auto_key_setup=False
)

## Key Control

### Generate Keys

To use enVector, we generate the FHE key bundle, which provides the cryptographic basis for ciphertexts in fully homomorphic encryption (FHE). With KMS enabled, `ev.generate_key(key_id=...)` asks the **KMS to generate the bundle** and identifies it by `key_id`:
- **Secret key** — used to decrypt scores. Generated and held **inside the KMS**; it never reaches the SDK or the enVector server.
- **Encryption key** — used by the client to encrypt vectors and queries. Synced from the KMS to the SDK.
- **Evaluation key** — uploaded to the enVector server so it can perform homomorphic operations on ciphertexts (but cannot decrypt them).

When metadata is sensitive, set `metadata_encryption=True` so the KMS also manages a separate AES-256 metadata key, which likewise stays inside the KMS.

The `eval_mode`/`preset` pair selects the homomorphic evaluation scheme and must match the KMS server configuration; this example uses `mm32` (preset `ip2`).

For a more detailed description, see [Key Generation](https://docs.envector.io/user-guide/advanced-user-guide/key-generation) in [enVector Documents](https://docs.envector.io/).

In [ ]:
ev.generate_key(key_id=key_id)

### Register Key

Register the key with the enVector server.
`register_key()` uploads **only** the evaluation key; the secret key and the AES metadata key stay inside the **KMS** and are never uploaded.
With just the evaluation key the server can perform homomorphic operations on ciphertexts but cannot decrypt them.

`load_key()` then selects this key as the active key for subsequent index operations on the server.

For more details, see [Key Registration](https://docs.envector.io/user-guide/connection/key-management#key-registration) in [enVector Documents](https://docs.envector.io/).

In [ ]:
# Register and load the key on the server
ev.register_key(key_id=key_id)
ev.load_key(key_id=key_id)

We can inspect the information about keys:
- `get_key_list()`: offers the list of registered key IDs
- `get_key_info()`: returns parameters such as preset, eval mode, and supported dimensions for a specific key.

In [ ]:
ev.get_key_list()

In [ ]:
ev.get_key_info(key_id)

## Prepare Index

### Create Index

Before inserting data, we create an `Index` with the dimension of the vectors to store.
`dim` must match the dimension of the vectors you plan to insert.

For more details, see [Creation](https://docs.envector.io/user-guide/encrypted-index/creation) in [enVector Documents](https://docs.envector.io/).

In [ ]:
index_name = "e2e_index"
dim = 512

In [ ]:
# Create an index bound to this key config
index = ev.create_index(index_name, dim=dim)

In [ ]:
index.index_config

### Prepare data

Here we prepare sample vectors and matching metadata to insert into the enVector server.
Vectors must be L2-normalized before insertion.
Metadata can be any JSON-serializable value (string, dict, list, etc.).

In [ ]:
import numpy as np

# Generate sample normalized vectors and metadata
vecs = np.random.rand(10, dim)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
metadata = [f"Item {i+1}" for i in range(10)]

### Encrypt Items

Before sending data to the server, we encrypt vectors and metadata on the client side.

**Vectors (FHE):** the `Cipher` class encrypts vectors and queries with the encryption key synced from the KMS. `index.cipher.encrypt(vecs, "item")` encodes the vectors for insertion and returns a `CipherBlock`. Score decryption is later delegated to the KMS — see the search section.

**Metadata (AES-256 via KMS):** when metadata is sensitive, encrypt each entry with `kms_client.encrypt_metadata(key_id, plaintext_metadata)`. The client sends plaintext and the KMS returns ciphertext bytes — the metadata key never leaves the KMS.

Note that `Index.insert()` can also accept plaintext directly and perform the encryption internally using the `Cipher` bound to the index — the explicit `encrypt` / `encrypt_metadata` calls below are shown to make the encryption step visible.

For more details, see [Encryption](https://docs.envector.io/user-guide/advanced-user-guide/cipher/encryption) in [enVector Documents](https://docs.envector.io/).

In [ ]:
from pyenvector.crypto import Cipher
from pyenvector import SealedBlob

# Explicitly encrypt each item (vector) before sending
cipher = Cipher(
    dim=dim,
    use_key_stream=True,
    enc_key=ev.pyenvector_client.index_config.enc_key,
    preset=preset,
    eval_mode=eval_mode,
)
enc_vectors = index.cipher.encrypt(vecs, "item")
print("Encrypted Vectors\n", enc_vectors)

# Encrypt metadata via KMS
enc_metadata = [
    SealedBlob(ct)
    for ct in ev.pyenvector_client.kms_client.encrypt_metadata(
        key_id=key_id,
        plaintext_metadata=metadata,
    )
]
print("Encrypted Metadata\n", enc_metadata)

### Insert the Encrypted Vectors

Now we insert the pre-encrypted vectors and metadata into the index on the enVector server.

To avoid double-encryption, pre-encrypted metadata bytes are wrapped in `SealedBlob`. The SDK stores them
as Base64 wire strings without an additional encryption pass.

`Index.insert()` is asynchronous on the server side, and a single insert flows through three server-side stages:
1. **flush** — the client splits the batch and the server persists each ciphertext shard.
2. **segmentation** — the server merges the persisted shards into a search-ready segment.
3. **load** — the merged segment is published so the index becomes searchable.

`wait_for_insert_stage` polls until the specified stage is reached, blocking the notebook until the data
is fully merged and searchable before proceeding to the search section.

For more details, see [Inserting Data](https://docs.envector.io/user-guide/encrypted-index/insertion) in [enVector Documents](https://docs.envector.io/).

In [ ]:
request_ids = []
# flush: split batch → server persists each shard; capture request_ids for tracking
index.insert(
    enc_vectors,
    enc_metadata,
    request_ids=request_ids,
    execute_until="flush",
    load=False,
    await_completion=False,
)
# segmentation: merge persisted shards into a search-ready segment
index.indexing(request_ids)
# load: publish the merged segment so the index becomes searchable
index.load()
# block until segmentation is complete before searching
index.wait_for_insert_stage(
    request_ids=request_ids,
    target_stage="segmentation",
    timeout_s=3000,
    poll_interval_s=1.0,
)

## Encrypted Similarity Search

### Prepare query

First, prepare a query vector. Because the index was configured with the default `query_encryption="plain"`, we can send the query vector as plaintext — the server still computes scores against the encrypted index and returns encrypted similarity scores.

In [ ]:
query = vecs[0]

### Encrypted search on the index

With the encrypted index loaded, we can perform similarity search directly over the ciphertexts.

The high-level `Index.search()` chains the underlying steps internally — `scoring` on the server, then score decryption, top-k metadata retrieval, and metadata decryption **through the KMS** — but here we run them explicitly to see what happens at every stage.

`Index.scoring()` is the homomorphic part: the server computes inner products between the query and every encrypted vector in the index and returns one encrypted score block per query as a list of `CipherBlock`.

For more details, see [Scoring](https://docs.envector.io/user-guide/search/scoring) in the [enVector Documents](https://docs.envector.io/).

In [ ]:
# Server performs encrypted similarity scoring; response is encrypted
encrypted_scores = index.scoring(query)
encrypted_scores

### Step 1 — Score decryption + top-k ranking (KMS)

`_kms_topk` sends the encrypted score block to the KMS. The KMS decrypts each score using the secret key
(which never leaves the KMS), computes the inner-product ranking, and returns the `top_k` best positions
as `TopKResult` objects — each carrying a plaintext `score` and a `metadata_idx` (shard + row address).


In [ ]:
search_index = ev.Index(index_name)
kms_client = ev.pyenvector_client.kms_client

# scoring() returns list[CipherBlock]; single-query → [0]
ranked = search_index._kms_topk(encrypted_scores[0], top_k=3)

for r in ranked:
    print(f"score={r.score:.6f}  shard={r.metadata_idx.shard_idx}  row={r.metadata_idx.row_idx}")

### Step 2 — Fetch encrypted metadata from the server

Using the `metadata_idx` positions returned by the KMS, the SDK fetches the corresponding metadata entries
directly from the enVector server. At this point the metadata is still a **Base64-encoded AES ciphertext** —
the server stored it encrypted and cannot read it. The secret key has not left the KMS.


In [ ]:
import base64

topk_indices = [
    {"shard_idx": r.metadata_idx.shard_idx, "row_idx": r.metadata_idx.row_idx}
    for r in ranked
]

raw_meta = search_index.indexer.get_metadata(index_name, topk_indices, fields=["metadata"])

enc_blobs = [base64.b64decode(item.data) for item in raw_meta if item.data]
print(f"{len(enc_blobs)} encrypted metadata blobs (still AES-ciphertext):")
for blob in enc_blobs:
    print(" ", blob[:32], "...")

### Step 3 — Metadata decryption via KMS

`kms_client.decrypt_metadata()` sends the encrypted blobs to the KMS, which decrypts them with the AES
metadata key and returns plaintext strings — without the secret key ever leaving the KMS.


In [ ]:
# KMS decrypts metadata on the client's behalf
plaintexts = kms_client.decrypt_metadata(key_id, enc_blobs)

import json as _json
results = []
for r, item, pt in zip(ranked, raw_meta, plaintexts):
    try:
        meta_value = _json.loads(pt)
    except Exception:
        meta_value = pt
    results.append({"id": item.id, "score": r.score, "metadata": meta_value})

print(*results, sep="\n")

### Clean up

When the index and key are no longer needed, drop the index from the server with `ev.drop_index()` and remove the registered key with `ev.delete_key()`. The key material itself is managed by the KMS; use the KMS key-lifecycle operations (see the `06-kms-key-lifecycle` notebook) to rotate, suspend, or destroy it.

In [ ]:
ev.drop_index(index_name)

In [ ]:
ev.delete_key(key_id)

For more detailed API reference, see [enVector Documents](https://docs.envector.io/).